In [ ]:
%run ../globalvariables

In [ ]:
%run ../lakehousefunction

In [ ]:
from pyspark.sql.functions import col, isnan, when

In [ ]:
NOTEBOOK = "silver/trafico"

In [ ]:
# Negatives and NaN to null
errors = []
success = False

try:
    trafico = spark.table(f"{BRONZE_TABLE}.trafico")

    for metric in TRAFFIC_METRICS:
        trafico = trafico.withColumn(metric, col(metric).cast("double"))
        trafico = trafico.withColumn(
            metric,
            when((col(metric) < 0) | isnan(col(metric)), None).otherwise(col(metric)),
        )

    trafico = trafico.withColumn(
        "intensidad",
        when(col("intensidad") > INTENSIDAD_MAX, None).otherwise(col("intensidad")),
    )

    trafico = trafico.dropna(subset=["intensidad"]).dropDuplicates()

    if not write_silver(trafico, "trafico"):
        raise Exception("write_silver returned False")
    success = True
    print("trafico written")
except Exception as e:
    errors.append(error_record(NOTEBOOK, e))
    print(f"fail trafico: {type(e).__name__}: {e}")

In [ ]:
print(f"trafico: {'SUCCESS' if success else 'FAILED'}")
log_errors(errors)